In [1]:
%cd ..

/home/ai_n_zag@lab.graphicon.ru/tmp/kolobok


In [2]:
from pathlib import Path
from typing import Tuple, Optional, Iterable, Callable
import shutil

import cv2
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    swin_s,
    Swin_S_Weights,
    efficientnet_b7,
    EfficientNet_B7_Weights,
    swin_v2_t,
    Swin_V2_T_Weights,
    vit_b_16,
    ViT_B_16_Weights,
)
from torchvision.io import read_image
from torchvision import transforms

from tqdm import tqdm

In [3]:
data_root = Path("data/dataset_crop")

df = pd.read_csv(data_root / "thread_depths.csv")

In [4]:
class ThreadDataset(Dataset):
    def __init__(self, data: pd.DataFrame, data_root_dir: str, transform: Optional[nn.Module] = None):
        self.data = data
        self.data_root_dir = Path(data_root_dir)
        self.image_paths = []
        self.labels = []
        for _, row in self.data.iterrows():
            image_path = self.data_root_dir / row["path"]
            if not image_path.exists():
                print(f"Warning: {image_path} does not exist")
                continue
            self.image_paths.append(self.data_root_dir / row["path"])
            self.labels.append(row["label"])

        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, float]:
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = read_image(str(image_path))
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [5]:
dataset = ThreadDataset(
    df, data_root, transform=Swin_V2_T_Weights.IMAGENET1K_V1.transforms()
)

In [6]:
class CropPadding(nn.Module):
    def __init__(self, num_pixels: int):
        super().__init__()
        self.num_pixels = num_pixels

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        *_, h, w = image.shape
        return image[
            ...,
            self.num_pixels : h - self.num_pixels,
            self.num_pixels : w - self.num_pixels,
        ]


augmentation_transform = transforms.Compose(
    [
        # transforms.Pad(100, padding_mode="reflect"),
        transforms.ToPILImage(),
        transforms.RandAugment(
            num_ops=4, interpolation=transforms.InterpolationMode.BICUBIC
        ),
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        # CropPadding(100),
    ]
)

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(2),
)
# train_dataset.dataset.transform = augmentation_transform

In [7]:
train_loader = DataLoader(train_dataset, shuffle=True, num_workers=8, batch_size=128)
val_loader = DataLoader(val_dataset, shuffle=False, num_workers=8)

In [8]:
# model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
# model.fc = nn.Sequential(nn.Linear(2048, 512), nn.ReLU(), nn.Linear(512, 1))

# model = swin_v2_s(weights=Swin_V2_S_Weights.IMAGENET1K_V1)
# model.head = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, 1))

# model = swin_s(weights=Swin_S_Weights.IMAGENET1K_V1)
# model.head = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, 1))

# model = efficientnet_b7(weights=EfficientNet_B7_Weights.IMAGENET1K_V1)
# model.classifier = nn.Sequential(nn.Linear(2560, 512), nn.SiLU(), nn.Linear(512, 1))

model = swin_v2_t(weights=Swin_V2_T_Weights.IMAGENET1K_V1)
model.head = nn.Sequential(nn.Linear(768, 512), nn.GELU(), nn.Linear(512, 1))

# model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_SWAG_LINEAR_V1)
# model.heads = nn.Sequential(nn.Linear(768, 512), nn.GELU(), nn.Linear(512, 1))

In [9]:
class Lion(optim.Optimizer):
    def __init__(
        self,
        params: Iterable[nn.Parameter],
        lr: float = 1e-4,
        beta1: float = 0.9,
        beta2: float = 0.99,
        weight_decay: float = 0,
    ):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, weight_decay=weight_decay)
        super().__init__(params, defaults)

    def step(self, closure: Callable[[], None] = None):
        for group in self.param_groups:
            lr = group["lr"]
            beta1 = group["beta1"]
            beta2 = group["beta2"]
            lambda_ = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                grad = p.grad.data
                state = self.state.setdefault(p, {})

                if "exp_avg" not in state:
                    state["exp_avg"] = torch.zeros_like(p.data, device=p.device)

                c = beta1 * state["exp_avg"] + (1 - beta1) * grad

                if lambda_ > 0:
                    p.data.add_(p.data, alpha=-lr * lambda_)

                p.data.add_(torch.sign(c), alpha=-lr)

                state["exp_avg"].mul_(beta2).add_(grad, alpha=1 - beta2)


In [10]:
def train_fn(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 10,
    grad_accumulation_steps: int = 1,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, (images, labels) in tqdm(
            enumerate(train_loader),
            desc=f"Epoch {epoch + 1} Training",
            total=len(train_loader),
        ):
            images = images.to(device)
            labels = labels.to(device, torch.float32).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(images).exp()
            loss = criterion(outputs, labels) / grad_accumulation_steps
            loss.backward()

            if (i + 1) % grad_accumulation_steps == 0:
                optimizer.step()

            running_loss += loss.item()

        print(
            f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}"
        )

        model.eval()
        mae = []
        within_tolerance = []
        with torch.no_grad():
            for images, labels in tqdm(
                val_loader, desc=f"Epoch {epoch + 1} Evalutaion"
            ):
                images = images.to(device)
                labels = labels.to(device).unsqueeze(1)

                outputs = model(images).exp()
                mae.append(torch.abs(outputs - labels).mean().item())
                within_tolerance.append(
                    int(torch.abs(outputs - labels).squeeze().item() <= 1)
                )
        print(
            f"Validation MAE: {np.mean(mae):.4f}, Fraction of errors <= 1: {np.mean(within_tolerance)}"
        )

In [ ]:
train_fn(
    model,
    train_loader,
    val_loader,
    num_epochs=10,
    grad_accumulation_steps=1,
)

Epoch 1 Training: 100%|██████████| 10/10 [00:12<00:00,  1.29s/it]


Epoch [1/10], Loss: 11.1822


Epoch 1 Evalutaion: 100%|██████████| 289/289 [00:04<00:00, 59.24it/s]

Validation MAE: 1.6976, Fraction of errors <= 1: 0.32525951557093424



Epoch 2 Training: 100%|██████████| 10/10 [00:12<00:00,  1.29s/it]


...


Epoch 7 Training: 100%|██████████| 10/10 [00:13<00:00,  1.31s/it]


Epoch [7/10], Loss: 1.2586


Epoch 7 Evalutaion: 100%|██████████| 289/289 [00:04<00:00, 58.64it/s]

Validation MAE: 1.0394, Fraction of errors <= 1: 0.5570934256055363



Epoch 8 Training:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# # img_path = "/home/ai_n_zag@lab.graphicon.ru/tmp/kolobok/data/threads/thread1.png"
# # img = read_image(img_path).permute(1, 2, 0).numpy()
# # # img = get_thread_crop(img)
# # img = torch.tensor(img).permute(2, 0, 1).to(torch.float32) / 255
# model.eval()
# with torch.no_grad():
#     for img, label in val_loader:
#         pred = model(img.cuda()).exp()
#         print(pred, label)  

tensor([[6.0771]], device='cuda:0') tensor([6.1000], dtype=torch.float64)
tensor([[3.1826]], device='cuda:0') tensor([3.2000], dtype=torch.float64)
tensor([[2.0493]], device='cuda:0') tensor([3.], dtype=torch.float64)
tensor([[6.4794]], device='cuda:0') tensor([6.1000], dtype=torch.float64)
tensor([[0.7433]], device='cuda:0') tensor([1.7000], dtype=torch.float64)
tensor([[5.3466]], device='cuda:0') tensor([7.2000], dtype=torch.float64)
tensor([[8.7228]], device='cuda:0') tensor([8.5000], dtype=torch.float64)
tensor([[5.6025]], device='cuda:0') tensor([7.3000], dtype=torch.float64)
tensor([[7.6769]], device='cuda:0') tensor([8.], dtype=torch.float64)
tensor([[2.0792]], device='cuda:0') tensor([1.4000], dtype=torch.float64)
tensor([[8.3070]], device='cuda:0') tensor([9.], dtype=torch.float64)
tensor([[1.4105]], device='cuda:0') tensor([3.2000], dtype=torch.float64)
tensor([[3.0110]], device='cuda:0') tensor([3.2000], dtype=torch.float64)
tensor([[5.5755]], device='cuda:0') tensor([6.8000